# Augmentation — Generate 500 More Bad Images

We have 1500 bad images. We need 2000 (to match good images).
So we generate **500 more** by applying degradations to existing bad images.

### What is Augmentation?
Augmentation means taking an existing image and applying changes to it (blur, rotate, darken etc.) to create a new training sample. The model sees more variety and becomes more robust.

### Why only augment bad images?
Because we already have 2000 good images. We just need to balance the bad class.

## 1. Imports

In [ ]:
import os
import random
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

random.seed(42)
print('Imports done ✅')

## 2. Paths

In [ ]:
BAD_IMAGES_DIR  = './data/raw/wider_face'   # your existing 1500 bad images
OUTPUT_DIR      = './data/augmented/bad'    # where new images will be saved
HOW_MANY        = 500                       # how many new images to generate

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output folder ready: {OUTPUT_DIR}')

## 3. Learn — What Each Augmentation Does

Before applying, let's understand each technique visually.

In [ ]:
# Pick any one image to demonstrate on
sample_path = random.choice([
    os.path.join(BAD_IMAGES_DIR, f)
    for f in os.listdir(BAD_IMAGES_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

original = Image.open(sample_path).convert('RGB').resize((224, 224))

# Define each augmentation separately so you can see what each one does
augmentations = {
    'Original'              : transforms.Compose([]),

    # Makes image blurry — simulates motion blur or out-of-focus face
    'GaussianBlur'          : transforms.GaussianBlur(kernel_size=11, sigma=(3, 5)),

    # Rotates the face — simulates side profile or tilted head
    'Rotation (60°)'        : transforms.RandomRotation(degrees=(50, 70)),

    # Makes image very dark — simulates poor lighting
    'Dark (brightness)'     : transforms.ColorJitter(brightness=(0.1, 0.3)),

    # Randomly cuts part of the image — simulates occlusion (hand over face, mask etc.)
    'RandomCrop'            : transforms.RandomCrop(size=(160, 160)),

    # Flips horizontally — different angle of same bad face
    'HorizontalFlip'        : transforms.RandomHorizontalFlip(p=1.0),

    # Reduces contrast — washed out or overexposed image
    'Low Contrast'          : transforms.ColorJitter(contrast=(0.1, 0.3)),
}

# Show all augmentations side by side
fig, axes = plt.subplots(1, len(augmentations), figsize=(20, 4))
fig.suptitle('Augmentation Techniques — Each Creates a New Bad Sample', fontsize=13, fontweight='bold')

for ax, (name, transform) in zip(axes, augmentations.items()):
    augmented = transform(original)
    ax.imshow(augmented)
    ax.set_title(name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Define the Augmentation Pipeline

Now we combine them into one pipeline.
`RandomApply` means each augmentation has a chance of being applied — so every generated image looks slightly different.

In [ ]:
augment = transforms.Compose([
    transforms.Resize((224, 224)),

    # Each line below = one technique, applied randomly
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=11, sigma=(2, 5))], p=0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=60)],                 p=0.5),
    transforms.RandomApply([transforms.ColorJitter(brightness=(0.1, 0.3))],         p=0.5),
    transforms.RandomApply([transforms.ColorJitter(contrast=(0.1, 0.3))],           p=0.3),
    transforms.RandomHorizontalFlip(p=0.5),
])

print('Augmentation pipeline ready ✅')
print('Each image will get a random mix of the above techniques')

## 5. Preview — What Generated Images Will Look Like

In [ ]:
# Apply the pipeline 8 times to the same image — each result is different
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('8 Augmented Versions of the Same Image', fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    augmented = augment(original)
    ax.imshow(augmented)
    ax.set_title(f'Generated #{i+1}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Generate & Save 500 New Bad Images

In [ ]:
# Collect all existing bad image paths
bad_images = [
    os.path.join(BAD_IMAGES_DIR, f)
    for f in os.listdir(BAD_IMAGES_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

print(f'Existing bad images : {len(bad_images)}')
print(f'Generating          : {HOW_MANY} new images')
print(f'Saving to           : {OUTPUT_DIR}')
print('─' * 40)

generated = 0

while generated < HOW_MANY:
    # Pick a random bad image each time
    src_path = random.choice(bad_images)

    try:
        img = Image.open(src_path).convert('RGB')
        augmented_img = augment(img)

        # Save with a clear name so we know it's augmented
        save_path = os.path.join(OUTPUT_DIR, f'aug_bad_{generated + 1:04d}.jpg')
        augmented_img.save(save_path, quality=95)

        generated += 1

        # Print progress every 100 images
        if generated % 100 == 0:
            print(f'  Generated {generated} / {HOW_MANY}')

    except Exception as e:
        print(f'  Skipped one image: {e}')
        continue

print(f'\n✅ Done! {generated} augmented bad images saved to {OUTPUT_DIR}')

## 7. Verify Final Count

In [ ]:
original_bad  = len(bad_images)
augmented_bad = len(os.listdir(OUTPUT_DIR))
total_bad     = original_bad + augmented_bad

print(f'Original bad images   : {original_bad}')
print(f'Augmented bad images  : {augmented_bad}')
print(f'Total bad images      : {total_bad}')
print(f'Good images           : 2000')
print('─' * 35)
print(f'Dataset is balanced   : {total_bad == 2000} ✅')

---
## ✅ What's Done / What's Next

| Step | Status |
|------|--------|
| 2000 good images (FFHQ) | ✅ Done |
| 1500 bad images (WIDER FACE) | ✅ Done |
| 500 augmented bad images | ✅ Done |
| Total: 2000 good / 2000 bad (balanced) | ✅ Done |
| Run RetinaFace → crop all faces to 112×112 | ⬜ Next notebook |